# Transfer Learning
Перенос знаний - это подход в машинном обучении, при котором готовая модель, обученная на одной задаче, используется как база для решения другой, похожей задачи

In [38]:
import torch
from torch import nn
from torch.utils.data import DataLoader
import torchvision
from torchvision import transforms, datasets
import torchvision.models as models
from tqdm import tqdm
import time

In [39]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [40]:
import urllib.request, zipfile

url = "https://download.pytorch.org/tutorial/hymenoptera_data.zip"
urllib.request.urlretrieve(url, "hymenoptera_data.zip")

with zipfile.ZipFile("hymenoptera_data.zip") as z:
    z.extractall(".")

In [41]:
transforms_train = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

transforms_val = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

In [42]:
train = datasets.ImageFolder('/content/hymenoptera_data/train', transform=transforms_train)
val = datasets.ImageFolder('/content/hymenoptera_data/val', transform=transforms_val)

In [43]:
class_names = train.classes
class_names

['ants', 'bees']

In [44]:
print(len(train), len(val))

244 153


In [45]:
train_loader = DataLoader(train, batch_size=16,shuffle=True)
val_loader = DataLoader(val, batch_size=16)

## Обучение с нуля, без Transfer Learning

### Вариант 1:
* Модель не обучена
* Меняем только последний слой

In [46]:
model = models.vgg11()
model

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU(inplace=True)
    (10): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (11): Conv2d(256, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (12): ReLU(inplace=True)
    (13): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (14): ReLU(inplace=True)
    (15): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
 

In [47]:
model.classifier

Sequential(
  (0): Linear(in_features=25088, out_features=4096, bias=True)
  (1): ReLU(inplace=True)
  (2): Dropout(p=0.5, inplace=False)
  (3): Linear(in_features=4096, out_features=4096, bias=True)
  (4): ReLU(inplace=True)
  (5): Dropout(p=0.5, inplace=False)
  (6): Linear(in_features=4096, out_features=1000, bias=True)
)

Перезаписываем последний слой сети, чтобы он выводил только 2 класса

In [48]:
model.classifier[6] = nn.Linear(4096,2)

In [49]:
model = model.to(device)

loss_model = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=0.001)
lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=5)

Тренировка модели

In [50]:
EPOCHS = 10

train_loss = []
val_loss = []
train_acc = []
val_acc = []
lr_list = []
best_loss = None
count = 0 # число операций без улучшения модели

start = time.time()

for epoch in range(EPOCHS):

    # Тренировка модели
    model.train()
    running_train_loss = []
    true_answer = 0
    train_loop = tqdm(train_loader, leave=False)
    for x, targets in train_loop:
        # Данные
        # (batch_size, 1, 28, 28) -> (batch_size, 784)
        # x = x.reshape(-1, 28*28).to(device)
        x = x.to(device)
        # (batch_size, int) -> (batch_size, 10), dtype=float32
        targets = targets.reshape(-1).to(torch.int32)
        targets = torch.eye(2)[targets].to(device)

        # Прямой проход + расчет ошибки модели
        pred = model(x)
        loss = loss_model(pred, targets)

        # Обратный проход
        opt.zero_grad()
        loss.backward()
        # Шаг оптимизации
        opt.step()

        running_train_loss.append(loss.item())
        mean_train_loss = sum(running_train_loss)/len(running_train_loss)

        true_answer += (pred.argmax(dim=1) == targets.argmax(dim=1)).sum().item()

        train_loop.set_description(f"Epoch [{epoch+1}/{EPOCHS}], train_loss={mean_train_loss:.4f}")

    # Расчет значения метрики
    running_train_acc = true_answer / len(train)

    # Сохранение значения функции потерь и метрики
    train_loss.append(mean_train_loss)
    train_acc.append(running_train_acc)

    # Проверка модели (валидация)
    model.eval()
    with torch.no_grad():
        running_val_loss = []
        true_answer = 0
        for x, targets in val_loader:
            # Данные
            # (batch_size, 1, 28, 28) -> (batch_size, 784)
            # x = x.reshape(-1, 28*28).to(device)
            x = x.to(device)
            # (batch_size, int) -> (batch_size, 10), dtype=float32
            targets = targets.reshape(-1).to(torch.int32)
            targets = torch.eye(2)[targets].to(device)

            # Прямой проход + расчет ошибки модели
            pred = model(x)
            loss = loss_model(pred, targets)

            running_val_loss.append(loss.item())
            mean_val_loss = sum(running_val_loss)/len(running_val_loss)

            true_answer += (pred.argmax(dim=1) == targets.argmax(dim=1)).sum().item()

        # Расчет значения метрики
        running_val_acc = true_answer / len(val)

        # Сохранение значения функции потерь и метрики
        val_loss.append(mean_val_loss)
        val_acc.append(running_val_acc)

        lr_scheduler.step(mean_val_loss)
        lr = lr_scheduler._last_lr[0]
        lr_list.append(lr)

        print(f"Epoch [{epoch+1}/{EPOCHS}], train_loss={mean_train_loss:.4f}, train_acc={running_train_acc:.4f}, val_loss={mean_val_loss:.4f}, val_acc={running_val_acc:.4f}, lr={lr}")

finish = time.time() - start
print(f'Время обучения {EPOCHS} эпох: {finish//60:.0f}мин {finish%60:.0f}сек')

Epoch [1/10], train_loss=7.4736, train_acc=0.4590, val_loss=0.6929, val_acc=0.5425, lr=0.001


Epoch [2/10], train_loss=0.6935, train_acc=0.4959, val_loss=0.6958, val_acc=0.4575, lr=0.001


Epoch [3/10], train_loss=0.6971, train_acc=0.5041, val_loss=0.6933, val_acc=0.4379, lr=0.001


Epoch [4/10], train_loss=0.6945, train_acc=0.4918, val_loss=0.6918, val_acc=0.5425, lr=0.001


Epoch [5/10], train_loss=0.6939, train_acc=0.4549, val_loss=0.6936, val_acc=0.4379, lr=0.001


Epoch [6/10], train_loss=0.6930, train_acc=0.5205, val_loss=0.6907, val_acc=0.5425, lr=0.001


Epoch [7/10], train_loss=0.6946, train_acc=0.5041, val_loss=0.6899, val_acc=0.5490, lr=0.001


Epoch [8/10], train_loss=0.6838, train_acc=0.5738, val_loss=0.6774, val_acc=0.5294, lr=0.001


Epoch [9/10], train_loss=0.6761, train_acc=0.5697, val_loss=0.7421, val_acc=0.4575, lr=0.001


Epoch [10/10], train_loss=0.6908, train_acc=0.5820, val_loss=0.6622, val_acc=0.5948, lr=0.001
Время обучения 10 эпох: 0мин 52сек


### Вариант 2
* Модель не обучена
* Меняем весь классификатор

In [51]:
model = models.vgg11()
model.classifier = nn.Linear(512*7*7,2)
model

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU(inplace=True)
    (10): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (11): Conv2d(256, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (12): ReLU(inplace=True)
    (13): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (14): ReLU(inplace=True)
    (15): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
 

In [52]:
model = model.to(device)

loss_model = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=0.001)
lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=5)

In [53]:
EPOCHS = 10

train_loss = []
val_loss = []
train_acc = []
val_acc = []
lr_list = []
best_loss = None
count = 0 # число операций без улучшения модели

start = time.time()

for epoch in range(EPOCHS):

    # Тренировка модели
    model.train()
    running_train_loss = []
    true_answer = 0
    train_loop = tqdm(train_loader, leave=False)
    for x, targets in train_loop:
        # Данные
        # (batch_size, 1, 28, 28) -> (batch_size, 784)
        # x = x.reshape(-1, 28*28).to(device)
        x = x.to(device)
        # (batch_size, int) -> (batch_size, 10), dtype=float32
        targets = targets.reshape(-1).to(torch.int32)
        targets = torch.eye(2)[targets].to(device)

        # Прямой проход + расчет ошибки модели
        pred = model(x)
        loss = loss_model(pred, targets)

        # Обратный проход
        opt.zero_grad()
        loss.backward()
        # Шаг оптимизации
        opt.step()

        running_train_loss.append(loss.item())
        mean_train_loss = sum(running_train_loss)/len(running_train_loss)

        true_answer += (pred.argmax(dim=1) == targets.argmax(dim=1)).sum().item()

        train_loop.set_description(f"Epoch [{epoch+1}/{EPOCHS}], train_loss={mean_train_loss:.4f}")

    # Расчет значения метрики
    running_train_acc = true_answer / len(train)

    # Сохранение значения функции потерь и метрики
    train_loss.append(mean_train_loss)
    train_acc.append(running_train_acc)

    # Проверка модели (валидация)
    model.eval()
    with torch.no_grad():
        running_val_loss = []
        true_answer = 0
        for x, targets in val_loader:
            # Данные
            # (batch_size, 1, 28, 28) -> (batch_size, 784)
            # x = x.reshape(-1, 28*28).to(device)
            x = x.to(device)
            # (batch_size, int) -> (batch_size, 10), dtype=float32
            targets = targets.reshape(-1).to(torch.int32)
            targets = torch.eye(2)[targets].to(device)

            # Прямой проход + расчет ошибки модели
            pred = model(x)
            loss = loss_model(pred, targets)

            running_val_loss.append(loss.item())
            mean_val_loss = sum(running_val_loss)/len(running_val_loss)

            true_answer += (pred.argmax(dim=1) == targets.argmax(dim=1)).sum().item()

        # Расчет значения метрики
        running_val_acc = true_answer / len(val)

        # Сохранение значения функции потерь и метрики
        val_loss.append(mean_val_loss)
        val_acc.append(running_val_acc)

        lr_scheduler.step(mean_val_loss)
        lr = lr_scheduler._last_lr[0]
        lr_list.append(lr)

        print(f"Epoch [{epoch+1}/{EPOCHS}], train_loss={mean_train_loss:.4f}, train_acc={running_train_acc:.4f}, val_loss={mean_val_loss:.4f}, val_acc={running_val_acc:.4f}, lr={lr}")

finish = time.time() - start
print(f'Время обучения {EPOCHS} эпох: {finish//60:.0f}мин {finish%60:.0f}сек')

Epoch [1/10], train_loss=4.2088, train_acc=0.5451, val_loss=0.6952, val_acc=0.4575, lr=0.001


Epoch [2/10], train_loss=0.6949, train_acc=0.4672, val_loss=0.6929, val_acc=0.5163, lr=0.001


Epoch [3/10], train_loss=0.6930, train_acc=0.5123, val_loss=0.6956, val_acc=0.4575, lr=0.001


Epoch [4/10], train_loss=0.6945, train_acc=0.5041, val_loss=0.6963, val_acc=0.4575, lr=0.001


Epoch [5/10], train_loss=0.6931, train_acc=0.5041, val_loss=0.6944, val_acc=0.4575, lr=0.001


Epoch [6/10], train_loss=0.6929, train_acc=0.5041, val_loss=0.6941, val_acc=0.4575, lr=0.001


Epoch [7/10], train_loss=0.6936, train_acc=0.5205, val_loss=0.7006, val_acc=0.4575, lr=0.001


Epoch [8/10], train_loss=0.6827, train_acc=0.5697, val_loss=0.6721, val_acc=0.5425, lr=0.001


Epoch [9/10], train_loss=0.7205, train_acc=0.5492, val_loss=0.6945, val_acc=0.4575, lr=0.001


Epoch [10/10], train_loss=0.6933, train_acc=0.5451, val_loss=0.6956, val_acc=0.4706, lr=0.001
Время обучения 10 эпох: 0мин 45сек


## Обучение с Transfer Learning

# Вариант 1
* Модель обучена
* Не замораживаем обученные параметры
* Меняем последний слой

In [54]:
model = models.vgg11(weights='DEFAULT')
model.classifier[6] = nn.Linear(4096, 2)
model

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU(inplace=True)
    (10): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (11): Conv2d(256, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (12): ReLU(inplace=True)
    (13): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (14): ReLU(inplace=True)
    (15): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
 

In [55]:
model = model.to(device)
loss_model = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=0.001)
lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=5)

 Из-за необученного последнего слоя будет получаться большая ошибка, из-за которой будут большие значения градиентов, которые побегут по всей сети и после первого шага сети обученные параметры станут необученными.

 Тем не менее, этот вариант по точности будет лучше предыдущих

In [56]:
EPOCHS = 10

train_loss = []
val_loss = []
train_acc = []
val_acc = []
lr_list = []
best_loss = None
count = 0 # число операций без улучшения модели

start = time.time()

for epoch in range(EPOCHS):

    # Тренировка модели
    model.train()
    running_train_loss = []
    true_answer = 0
    train_loop = tqdm(train_loader, leave=False)
    for x, targets in train_loop:
        # Данные
        # (batch_size, 1, 28, 28) -> (batch_size, 784)
        # x = x.reshape(-1, 28*28).to(device)
        x = x.to(device)
        # (batch_size, int) -> (batch_size, 10), dtype=float32
        targets = targets.reshape(-1).to(torch.int32)
        targets = torch.eye(2)[targets].to(device)

        # Прямой проход + расчет ошибки модели
        pred = model(x)
        loss = loss_model(pred, targets)

        # Обратный проход
        opt.zero_grad()
        loss.backward()
        # Шаг оптимизации
        opt.step()

        running_train_loss.append(loss.item())
        mean_train_loss = sum(running_train_loss)/len(running_train_loss)

        true_answer += (pred.argmax(dim=1) == targets.argmax(dim=1)).sum().item()

        train_loop.set_description(f"Epoch [{epoch+1}/{EPOCHS}], train_loss={mean_train_loss:.4f}")

    # Расчет значения метрики
    running_train_acc = true_answer / len(train)

    # Сохранение значения функции потерь и метрики
    train_loss.append(mean_train_loss)
    train_acc.append(running_train_acc)

    # Проверка модели (валидация)
    model.eval()
    with torch.no_grad():
        running_val_loss = []
        true_answer = 0
        for x, targets in val_loader:
            # Данные
            # (batch_size, 1, 28, 28) -> (batch_size, 784)
            # x = x.reshape(-1, 28*28).to(device)
            x = x.to(device)
            # (batch_size, int) -> (batch_size, 10), dtype=float32
            targets = targets.reshape(-1).to(torch.int32)
            targets = torch.eye(2)[targets].to(device)

            # Прямой проход + расчет ошибки модели
            pred = model(x)
            loss = loss_model(pred, targets)

            running_val_loss.append(loss.item())
            mean_val_loss = sum(running_val_loss)/len(running_val_loss)

            true_answer += (pred.argmax(dim=1) == targets.argmax(dim=1)).sum().item()

        # Расчет значения метрики
        running_val_acc = true_answer / len(val)

        # Сохранение значения функции потерь и метрики
        val_loss.append(mean_val_loss)
        val_acc.append(running_val_acc)

        lr_scheduler.step(mean_val_loss)
        lr = lr_scheduler._last_lr[0]
        lr_list.append(lr)

        print(f"Epoch [{epoch+1}/{EPOCHS}], train_loss={mean_train_loss:.4f}, train_acc={running_train_acc:.4f}, val_loss={mean_val_loss:.4f}, val_acc={running_val_acc:.4f}, lr={lr}")

finish = time.time() - start
print(f'Время обучения {EPOCHS} эпох: {finish//60:.0f}мин {finish%60:.0f}сек')

Epoch [1/10], train_loss=2.2145, train_acc=0.4877, val_loss=0.6969, val_acc=0.5033, lr=0.001


Epoch [2/10], train_loss=0.8921, train_acc=0.4631, val_loss=0.7604, val_acc=0.4575, lr=0.001


Epoch [3/10], train_loss=0.7833, train_acc=0.4631, val_loss=0.7151, val_acc=0.5033, lr=0.001


Epoch [4/10], train_loss=0.7690, train_acc=0.5246, val_loss=0.7071, val_acc=0.4641, lr=0.001


Epoch [5/10], train_loss=0.6849, train_acc=0.5369, val_loss=0.6751, val_acc=0.5686, lr=0.001


Epoch [6/10], train_loss=0.7187, train_acc=0.5041, val_loss=0.7552, val_acc=0.4575, lr=0.001


Epoch [7/10], train_loss=0.6988, train_acc=0.5533, val_loss=0.6563, val_acc=0.6340, lr=0.001


Epoch [8/10], train_loss=0.6253, train_acc=0.7008, val_loss=0.7558, val_acc=0.5490, lr=0.001


Epoch [9/10], train_loss=0.6078, train_acc=0.6844, val_loss=0.8017, val_acc=0.4771, lr=0.001


Epoch [10/10], train_loss=0.6664, train_acc=0.5902, val_loss=0.7288, val_acc=0.4967, lr=0.001
Время обучения 10 эпох: 0мин 54сек


### Вариант 2
* Модель обучена
* Не замораживаем обученные параметры
* Меняем весь классификатор

In [57]:
model = models.vgg11(weights='DEFAULT')
model.classifier = nn.Linear(512*7*7, 2)
model

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU(inplace=True)
    (10): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (11): Conv2d(256, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (12): ReLU(inplace=True)
    (13): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (14): ReLU(inplace=True)
    (15): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
 

In [58]:
model = model.to(device)
loss_model = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=0.001)
lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=5)

In [59]:
EPOCHS = 10

train_loss = []
val_loss = []
train_acc = []
val_acc = []
lr_list = []
best_loss = None
count = 0 # число операций без улучшения модели

start = time.time()

for epoch in range(EPOCHS):

    # Тренировка модели
    model.train()
    running_train_loss = []
    true_answer = 0
    train_loop = tqdm(train_loader, leave=False)
    for x, targets in train_loop:
        # Данные
        # (batch_size, 1, 28, 28) -> (batch_size, 784)
        # x = x.reshape(-1, 28*28).to(device)
        x = x.to(device)
        # (batch_size, int) -> (batch_size, 10), dtype=float32
        targets = targets.reshape(-1).to(torch.int32)
        targets = torch.eye(2)[targets].to(device)

        # Прямой проход + расчет ошибки модели
        pred = model(x)
        loss = loss_model(pred, targets)

        # Обратный проход
        opt.zero_grad()
        loss.backward()
        # Шаг оптимизации
        opt.step()

        running_train_loss.append(loss.item())
        mean_train_loss = sum(running_train_loss)/len(running_train_loss)

        true_answer += (pred.argmax(dim=1) == targets.argmax(dim=1)).sum().item()

        train_loop.set_description(f"Epoch [{epoch+1}/{EPOCHS}], train_loss={mean_train_loss:.4f}")

    # Расчет значения метрики
    running_train_acc = true_answer / len(train)

    # Сохранение значения функции потерь и метрики
    train_loss.append(mean_train_loss)
    train_acc.append(running_train_acc)

    # Проверка модели (валидация)
    model.eval()
    with torch.no_grad():
        running_val_loss = []
        true_answer = 0
        for x, targets in val_loader:
            # Данные
            # (batch_size, 1, 28, 28) -> (batch_size, 784)
            # x = x.reshape(-1, 28*28).to(device)
            x = x.to(device)
            # (batch_size, int) -> (batch_size, 10), dtype=float32
            targets = targets.reshape(-1).to(torch.int32)
            targets = torch.eye(2)[targets].to(device)

            # Прямой проход + расчет ошибки модели
            pred = model(x)
            loss = loss_model(pred, targets)

            running_val_loss.append(loss.item())
            mean_val_loss = sum(running_val_loss)/len(running_val_loss)

            true_answer += (pred.argmax(dim=1) == targets.argmax(dim=1)).sum().item()

        # Расчет значения метрики
        running_val_acc = true_answer / len(val)

        # Сохранение значения функции потерь и метрики
        val_loss.append(mean_val_loss)
        val_acc.append(running_val_acc)

        lr_scheduler.step(mean_val_loss)
        lr = lr_scheduler._last_lr[0]
        lr_list.append(lr)

        print(f"Epoch [{epoch+1}/{EPOCHS}], train_loss={mean_train_loss:.4f}, train_acc={running_train_acc:.4f}, val_loss={mean_val_loss:.4f}, val_acc={running_val_acc:.4f}, lr={lr}")

finish = time.time() - start
print(f'Время обучения {EPOCHS} эпох: {finish//60:.0f}мин {finish%60:.0f}сек')

Epoch [1/10], train_loss=0.8932, train_acc=0.4836, val_loss=0.6720, val_acc=0.5752, lr=0.001


Epoch [2/10], train_loss=0.7364, train_acc=0.5041, val_loss=0.6922, val_acc=0.4575, lr=0.001


Epoch [3/10], train_loss=0.6788, train_acc=0.5779, val_loss=1.2363, val_acc=0.5425, lr=0.001


Epoch [4/10], train_loss=0.7675, train_acc=0.5615, val_loss=0.6931, val_acc=0.5359, lr=0.001


Epoch [5/10], train_loss=0.6928, train_acc=0.5697, val_loss=0.6904, val_acc=0.5621, lr=0.001


Epoch [6/10], train_loss=0.6927, train_acc=0.5082, val_loss=0.6932, val_acc=0.4575, lr=0.001


Epoch [7/10], train_loss=0.6900, train_acc=0.5697, val_loss=0.6816, val_acc=0.5229, lr=0.0001


Epoch [8/10], train_loss=0.6785, train_acc=0.5164, val_loss=0.6718, val_acc=0.5686, lr=0.0001


Epoch [9/10], train_loss=0.6669, train_acc=0.5451, val_loss=0.6565, val_acc=0.5556, lr=0.0001


Epoch [10/10], train_loss=0.6509, train_acc=0.5943, val_loss=0.6442, val_acc=0.5621, lr=0.0001
Время обучения 10 эпох: 0мин 45сек


Результат все еще плохой

### Вариант 3
* Модель обучена
* Замораживаем обученные параметры
* Меняем весь классификатор

In [60]:
model = models.vgg11(weights='DEFAULT')

for param in model.parameters():
    param.requires_grad = False

model.classifier = nn.Linear(512*7*7, 2)
model

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU(inplace=True)
    (10): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (11): Conv2d(256, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (12): ReLU(inplace=True)
    (13): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (14): ReLU(inplace=True)
    (15): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
 

Проверим, что слой features обучен и заморожен, а классификатор не обучен и не заморожен

In [61]:
for name, layer in model.named_modules():
    if isinstance(layer, (nn.Conv2d, nn.Linear)):
        print(name)
        for i, param in enumerate(layer.parameters()):
            if i==0:
                print(f'weights.requires_grad = {param.requires_grad}')
            else:
                print(f'bias.requires_grad = {param.requires_grad}', end='\n\n')

features.0
weights.requires_grad = False
bias.requires_grad = False

features.3
weights.requires_grad = False
bias.requires_grad = False

features.6
weights.requires_grad = False
bias.requires_grad = False

features.8
weights.requires_grad = False
bias.requires_grad = False

features.11
weights.requires_grad = False
bias.requires_grad = False

features.13
weights.requires_grad = False
bias.requires_grad = False

features.16
weights.requires_grad = False
bias.requires_grad = False

features.18
weights.requires_grad = False
bias.requires_grad = False

classifier
weights.requires_grad = True
bias.requires_grad = True



B оптимизатор передаем параметры классификатора, потому что именно его мы хотим обучить

In [62]:
model = model.to(device)
loss_model = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.classifier.parameters(), lr=0.001)
lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=5)

In [63]:
EPOCHS = 10

train_loss = []
val_loss = []
train_acc = []
val_acc = []
lr_list = []
best_loss = None
count = 0 # число операций без улучшения модели

start = time.time()

for epoch in range(EPOCHS):

    # Тренировка модели
    model.train()
    running_train_loss = []
    true_answer = 0
    train_loop = tqdm(train_loader, leave=False)
    for x, targets in train_loop:
        # Данные
        # (batch_size, 1, 28, 28) -> (batch_size, 784)
        # x = x.reshape(-1, 28*28).to(device)
        x = x.to(device)
        # (batch_size, int) -> (batch_size, 10), dtype=float32
        targets = targets.reshape(-1).to(torch.int32)
        targets = torch.eye(2)[targets].to(device)

        # Прямой проход + расчет ошибки модели
        pred = model(x)
        loss = loss_model(pred, targets)

        # Обратный проход
        opt.zero_grad()
        loss.backward()
        # Шаг оптимизации
        opt.step()

        running_train_loss.append(loss.item())
        mean_train_loss = sum(running_train_loss)/len(running_train_loss)

        true_answer += (pred.argmax(dim=1) == targets.argmax(dim=1)).sum().item()

        train_loop.set_description(f"Epoch [{epoch+1}/{EPOCHS}], train_loss={mean_train_loss:.4f}")

    # Расчет значения метрики
    running_train_acc = true_answer / len(train)

    # Сохранение значения функции потерь и метрики
    train_loss.append(mean_train_loss)
    train_acc.append(running_train_acc)

    # Проверка модели (валидация)
    model.eval()
    with torch.no_grad():
        running_val_loss = []
        true_answer = 0
        for x, targets in val_loader:
            # Данные
            # (batch_size, 1, 28, 28) -> (batch_size, 784)
            # x = x.reshape(-1, 28*28).to(device)
            x = x.to(device)
            # (batch_size, int) -> (batch_size, 10), dtype=float32
            targets = targets.reshape(-1).to(torch.int32)
            targets = torch.eye(2)[targets].to(device)

            # Прямой проход + расчет ошибки модели
            pred = model(x)
            loss = loss_model(pred, targets)

            running_val_loss.append(loss.item())
            mean_val_loss = sum(running_val_loss)/len(running_val_loss)

            true_answer += (pred.argmax(dim=1) == targets.argmax(dim=1)).sum().item()

        # Расчет значения метрики
        running_val_acc = true_answer / len(val)

        # Сохранение значения функции потерь и метрики
        val_loss.append(mean_val_loss)
        val_acc.append(running_val_acc)

        lr_scheduler.step(mean_val_loss)
        lr = lr_scheduler._last_lr[0]
        lr_list.append(lr)

        print(f"Epoch [{epoch+1}/{EPOCHS}], train_loss={mean_train_loss:.4f}, train_acc={running_train_acc:.4f}, val_loss={mean_val_loss:.4f}, val_acc={running_val_acc:.4f}, lr={lr}")

finish = time.time() - start
print(f'Время обучения {EPOCHS} эпох: {finish//60:.0f}мин {finish%60:.0f}сек')

Epoch [1/10], train_loss=0.3775, train_acc=0.8566, val_loss=0.5011, val_acc=0.9281, lr=0.001


Epoch [2/10], train_loss=0.1199, train_acc=0.9672, val_loss=0.7800, val_acc=0.9150, lr=0.001


Epoch [3/10], train_loss=0.0835, train_acc=0.9590, val_loss=0.6249, val_acc=0.8889, lr=0.001


Epoch [4/10], train_loss=0.1478, train_acc=0.9754, val_loss=0.7597, val_acc=0.9150, lr=0.001


Epoch [5/10], train_loss=0.1271, train_acc=0.9590, val_loss=0.6804, val_acc=0.9216, lr=0.001


Epoch [6/10], train_loss=0.0843, train_acc=0.9631, val_loss=0.7884, val_acc=0.9216, lr=0.001


Epoch [7/10], train_loss=0.0646, train_acc=0.9795, val_loss=0.5905, val_acc=0.9281, lr=0.0001


Epoch [8/10], train_loss=0.0320, train_acc=0.9918, val_loss=0.6024, val_acc=0.9281, lr=0.0001


Epoch [9/10], train_loss=0.0107, train_acc=0.9918, val_loss=0.6026, val_acc=0.9281, lr=0.0001


Epoch [10/10], train_loss=0.0144, train_acc=0.9918, val_loss=0.5998, val_acc=0.9281, lr=0.0001
Время обучения 10 эпох: 0мин 36сек


Результат гораздо лучше, чем было

### Finetuning

In [64]:
model.features[13:] #Размораживаем несколько слоев для тренировки

Sequential(
  (13): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (14): ReLU(inplace=True)
  (15): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (16): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (17): ReLU(inplace=True)
  (18): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (19): ReLU(inplace=True)
  (20): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
)

In [65]:
for param in model.features[13:].parameters():
    param.requires_grad = True

In [66]:
for name, layer in model.named_modules():
    if isinstance(layer, (nn.Conv2d, nn.Linear)):
        print(name)
        for i, param in enumerate(layer.parameters()):
            if i==0:
                print(f'weights.requires_grad = {param.requires_grad}')
            else:
                print(f'bias.requires_grad = {param.requires_grad}', end='\n\n')

features.0
weights.requires_grad = False
bias.requires_grad = False

features.3
weights.requires_grad = False
bias.requires_grad = False

features.6
weights.requires_grad = False
bias.requires_grad = False

features.8
weights.requires_grad = False
bias.requires_grad = False

features.11
weights.requires_grad = False
bias.requires_grad = False

features.13
weights.requires_grad = True
bias.requires_grad = True

features.16
weights.requires_grad = True
bias.requires_grad = True

features.18
weights.requires_grad = True
bias.requires_grad = True

classifier
weights.requires_grad = True
bias.requires_grad = True



In [67]:
model = model.to(device)
loss_model = nn.CrossEntropyLoss()
opt = torch.optim.Adam([
    {'params': model.features[13:].parameters(), 'lr':0.000001},
    {'params': model.classifier.parameters()}
    ],
    lr = 0.001
)
lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=5)

In [68]:
EPOCHS = 10

train_loss = []
val_loss = []
train_acc = []
val_acc = []
lr_list = []
best_loss = None
count = 0 # число операций без улучшения модели

start = time.time()

for epoch in range(EPOCHS):

    # Тренировка модели
    model.train()
    running_train_loss = []
    true_answer = 0
    train_loop = tqdm(train_loader, leave=False)
    for x, targets in train_loop:
        # Данные
        # (batch_size, 1, 28, 28) -> (batch_size, 784)
        # x = x.reshape(-1, 28*28).to(device)
        x = x.to(device)
        # (batch_size, int) -> (batch_size, 10), dtype=float32
        targets = targets.reshape(-1).to(torch.int32)
        targets = torch.eye(2)[targets].to(device)

        # Прямой проход + расчет ошибки модели
        pred = model(x)
        loss = loss_model(pred, targets)

        # Обратный проход
        opt.zero_grad()
        loss.backward()
        # Шаг оптимизации
        opt.step()

        running_train_loss.append(loss.item())
        mean_train_loss = sum(running_train_loss)/len(running_train_loss)

        true_answer += (pred.argmax(dim=1) == targets.argmax(dim=1)).sum().item()

        train_loop.set_description(f"Epoch [{epoch+1}/{EPOCHS}], train_loss={mean_train_loss:.4f}")

    # Расчет значения метрики
    running_train_acc = true_answer / len(train)

    # Сохранение значения функции потерь и метрики
    train_loss.append(mean_train_loss)
    train_acc.append(running_train_acc)

    # Проверка модели (валидация)
    model.eval()
    with torch.no_grad():
        running_val_loss = []
        true_answer = 0
        for x, targets in val_loader:
            # Данные
            # (batch_size, 1, 28, 28) -> (batch_size, 784)
            # x = x.reshape(-1, 28*28).to(device)
            x = x.to(device)
            # (batch_size, int) -> (batch_size, 10), dtype=float32
            targets = targets.reshape(-1).to(torch.int32)
            targets = torch.eye(2)[targets].to(device)

            # Прямой проход + расчет ошибки модели
            pred = model(x)
            loss = loss_model(pred, targets)

            running_val_loss.append(loss.item())
            mean_val_loss = sum(running_val_loss)/len(running_val_loss)

            true_answer += (pred.argmax(dim=1) == targets.argmax(dim=1)).sum().item()

        # Расчет значения метрики
        running_val_acc = true_answer / len(val)

        # Сохранение значения функции потерь и метрики
        val_loss.append(mean_val_loss)
        val_acc.append(running_val_acc)

        lr_scheduler.step(mean_val_loss)
        lr = lr_scheduler._last_lr[0]
        lr_list.append(lr)

        print(f"Epoch [{epoch+1}/{EPOCHS}], train_loss={mean_train_loss:.4f}, train_acc={running_train_acc:.4f}, val_loss={mean_val_loss:.4f}, val_acc={running_val_acc:.4f}, lr={lr}")

finish = time.time() - start
print(f'Время обучения {EPOCHS} эпох: {finish//60:.0f}мин {finish%60:.0f}сек')

Epoch [1/10], train_loss=0.0165, train_acc=0.9918, val_loss=1.1828, val_acc=0.8954, lr=1e-06


Epoch [2/10], train_loss=0.1089, train_acc=0.9754, val_loss=1.0300, val_acc=0.9150, lr=1e-06


Epoch [3/10], train_loss=0.1193, train_acc=0.9795, val_loss=1.1920, val_acc=0.9085, lr=1e-06


Epoch [4/10], train_loss=0.0635, train_acc=0.9836, val_loss=0.8349, val_acc=0.9216, lr=1e-06


Epoch [5/10], train_loss=0.0154, train_acc=0.9918, val_loss=1.0299, val_acc=0.9150, lr=1e-06


Epoch [6/10], train_loss=0.0043, train_acc=1.0000, val_loss=0.9947, val_acc=0.9150, lr=1e-06


Epoch [7/10], train_loss=0.0279, train_acc=0.9959, val_loss=1.0600, val_acc=0.9216, lr=1e-06


Epoch [8/10], train_loss=0.0023, train_acc=1.0000, val_loss=0.9730, val_acc=0.9216, lr=1e-06


Epoch [9/10], train_loss=0.0002, train_acc=1.0000, val_loss=0.9836, val_acc=0.9216, lr=1e-06


Epoch [10/10], train_loss=0.0022, train_acc=1.0000, val_loss=0.8925, val_acc=0.9150, lr=1e-07
Время обучения 10 эпох: 0мин 36сек


Точность модели чуть-чуть выросла